In [ ]:
!nvidia-smi
!pip install torchmetrics[image] -q

In [ ]:
from __future__ import annotations
from typing import Callable, Iterable, Optional

import random
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch import Tensor
from torch.optim import Optimizer

from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.utils import make_grid

from torchmetrics.image import StructuralSimilarityIndexMeasure
from torchmetrics.image.fid import FrechetInceptionDistance

In [ ]:
SEED = 42 # For robust reporting, run multiple seeds: 42, 6, 11, 96, 22
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
# class HNAdam(Optimizer):
#     def __init__(
#         self,
#         params: Iterable[Tensor],
#         lr: float = 1e-3,
#         betas: tuple[float, float] = (0.9, 0.99),
#         eps: float = 1e-8,
#         lambda_t0: Optional[float] = None,
#     ) -> None:
#         if params is None:
#             raise ValueError("params cannot be None.")
#         if lr <= 0.0:
#             raise ValueError(f"Invalid learning rate: {lr}")
#         if eps < 0.0:
#             raise ValueError(f"Invalid epsilon value: {eps}")
#         if len(betas) != 2:
#             raise ValueError("betas must be a tuple of two floats")

#         beta1, beta2 = betas
#         if not 0.0 <= beta1 < 1.0:
#             raise ValueError(f"Invalid beta1 value: {beta1}")
#         if not 0.0 <= beta2 < 1.0:
#             raise ValueError(f"Invalid beta2 value: {beta2}")

#         if lambda_t0 is None:
#             lambda_t0 = random.uniform(2.0, 4.0)
#         if not 2.0 <= lambda_t0 <= 4.0:
#             raise ValueError(f"lambda_t0 must be in [2, 4], got {lambda_t0}")

#         defaults = {
#             "lr": lr,
#             "betas": (beta1, beta2),
#             "eps": eps,
#             "lambda_t0": lambda_t0,
#             "amsgrad": False,
#         }
#         super().__init__(params, defaults)

#         if len(self.param_groups) == 0:
#             raise ValueError("optimizer got an empty parameter list")

#     @torch.no_grad()
#     def step(self, closure: Optional[Callable[[], Tensor]] = None) -> Optional[Tensor]:
#         loss: Optional[Tensor] = None
#         if closure is not None:
#             with torch.enable_grad():
#                 loss = closure()

#         for group in self.param_groups:
#             lr: float = group["lr"]
#             beta1, beta2 = group["betas"]
#             eps: float = group["eps"]
#             lambda_t0: float = group["lambda_t0"]

#             for param in group["params"]:
#                 if param.grad is None:
#                     continue

#                 grad = param.grad
#                 if grad.is_sparse:
#                     raise RuntimeError("HNAdam does not support sparse gradients")

#                 state = self.state[param]

#                 if len(state) == 0:
#                     state["m"] = torch.zeros_like(param, memory_format=torch.preserve_format)
#                     state["v"] = torch.zeros_like(param, memory_format=torch.preserve_format)
#                     state["vhat"] = torch.zeros_like(param, memory_format=torch.preserve_format)

#                 m_prev: Tensor = state["m"]
#                 v_prev: Tensor = state["v"]
#                 vhat_prev: Tensor = state["vhat"]

#                 g_t = grad

#                 m_t = beta1 * m_prev + (1.0 - beta1) * g_t

#                 g_abs = g_t.abs()
#                 m_prev_norm = torch.linalg.vector_norm(m_prev)
#                 g_abs_norm = torch.linalg.vector_norm(g_abs)
#                 m_max = torch.maximum(m_prev_norm, g_abs_norm)

#                 zero = torch.zeros((), dtype=param.dtype, device=param.device)
#                 ratio = torch.where(m_max > 0.0, m_prev_norm / m_max, zero)
#                 lambda_t = torch.as_tensor(lambda_t0, dtype=param.dtype, device=param.device) - ratio

#                 v_t = beta2 * v_prev + (1.0 - beta2) * g_abs.pow(lambda_t)

#                 if bool((lambda_t < 2.0).item()):
#                     group["amsgrad"] = True

#                     vhat_t = torch.maximum(vhat_prev, v_t.abs())
#                     state["vhat"] = vhat_t

#                     denom = vhat_t.pow(1.0 / lambda_t) + eps
#                 else:
#                     group["amsgrad"] = False

#                     denom = v_t.pow(1.0 / lambda_t) + eps

#                 param.addcdiv_(m_t, denom, value=-lr)

#                 state["m"] = m_t
#                 state["v"] = v_t

#         return loss

In [ ]:
# Hyperparameters
input_dim = 3072 # 3 * 32 * 32 RGB images flattened
hidden_dim = 500
latent_dim = 20
BATCH_SIZE = 128

# Data loader
transform = transforms.ToTensor()

train_dataset = datasets.CIFAR10(root='./data', train=True, transform=transform, download=True)
test_dataset = datasets.CIFAR10(root='./data', train=False, transform=transform, download=True)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

class_names = train_dataset.classes

print(f'Train samples: {len(train_dataset)}')
print(f'Test samples:  {len(test_dataset)}')

# Display one sample image from each class
fig, axes = plt.subplots(1, 10, figsize=(18, 3))
class_samples = {}

for imgs, labels in train_loader:
    for i in range(imgs.size(0)):
        label = labels[i].item()
        if label not in class_samples:
            class_samples[label] = imgs[i]
        if len(class_samples) == 10:
            break
    if len(class_samples) == 10:
        break

for i in range(10):
    axes[i].imshow(np.transpose(class_samples[i].cpu().numpy(), (1, 2, 0)))
    axes[i].set_title(class_names[i], fontsize=9)
    axes[i].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Define the Encoder, Decoder, and VA
class Encoder(nn.Module):
    def __init__(self, input_dim, hidden_dim, latent_dim):
        super(Encoder, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc_mu = nn.Linear(hidden_dim, latent_dim)
        self.fc_logvar = nn.Linear(hidden_dim, latent_dim)

    def forward(self, x):
        h = torch.tanh(self.fc1(x))
        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)
        return mu, logvar


class Decoder(nn.Module):
    def __init__(self, latent_dim, hidden_dim, output_dim):
        super(Decoder, self).__init__()
        self.fc1 = nn.Linear(latent_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, output_dim)

    def forward(self, z):
        h = torch.tanh(self.fc1(z))
        x_hat = torch.sigmoid(self.fc2(h))
        return x_hat


class VAE(nn.Module):
    def __init__(self, input_dim, hidden_dim, latent_dim):
        super(VAE, self).__init__()
        self.encoder = Encoder(input_dim, hidden_dim, latent_dim)
        self.decoder = Decoder(latent_dim, hidden_dim, input_dim)

    def reparameterize(self, mu, logvar):
        # Reparameterization trick: z = mu + epsilon * std
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        z = mu + eps * std
        return z

    def forward(self, x):
        mu, logvar = self.encoder(x)
        z = self.reparameterize(mu, logvar)
        x_hat = self.decoder(z)
        return x_hat, mu, logvar


model = VAE(input_dim, hidden_dim, latent_dim).to(device)
model

In [ ]:
# Define the loss function
def vae_loss(x, x_hat, mu, logvar):
    # Reconstruction term (Bernoulli likelihood)
    BCE = F.binary_cross_entropy(x_hat, x, reduction='sum')

    # KL divergence between q(z|x) and p(z)=N(0,I)
    KLD = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())

    return BCE + KLD  

# Optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, betas=(0.9, 0.99), eps=1e-8)

In [ ]:
EPOCHS = 50
history = {
    'BCE + KLD': []
}

start_time = time.perf_counter()

for epoch in range(1, EPOCHS + 1):
    model.train()
    running_total = 0.0

    for x, _ in train_loader:
        x = x.to(device)
        x_flat = x.view(x.size(0), -1)

        optimizer.zero_grad()
        x_hat, mu, logvar = model(x_flat)
        total_loss = vae_loss(x_flat, x_hat, mu, logvar)
        total_loss.backward()
        optimizer.step()

        running_total += total_loss.item()

    avg_total = running_total / len(train_dataset)
    history['BCE + KLD'].append(avg_total)

    print(f'Epoch {epoch:02d}/{EPOCHS} | total={avg_total:.4f}')

training_time_sec = time.perf_counter() - start_time
print(f'Training completed in {training_time_sec:.2f} seconds')

In [ ]:
# Figure 1: loss minimization curve
epochs = np.arange(1, EPOCHS + 1)

plt.figure(figsize=(9, 5))
plt.plot(epochs, history['BCE + KLD'], label='Total loss', linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('Loss (average per sample)')
plt.title('Figure 1. VAE Loss Function Minimization Curve')
plt.grid(alpha=0.3)
plt.legend()
plt.show()

In [ ]:
@torch.no_grad()
def show_reconstructions(model, data_loader, n_images=10):
    model.eval()
    x, _ = next(iter(data_loader))
    x = x.to(device)
    x_flat = x.view(x.size(0), -1)

    x_prob, _, _ = model(x_flat)
    x_recon = x_prob.view(-1, 1, 28, 28)

    n = min(n_images, x.size(0))
    fig, axes = plt.subplots(2, n, figsize=(1.5 * n, 3.2))

    for i in range(n):
        axes[0, i].imshow(x[i, 0].cpu(), cmap='gray')
        axes[0, i].axis('off')
        axes[1, i].imshow(x_recon[i, 0].cpu(), cmap='gray')
        axes[1, i].axis('off')

    axes[0, 0].set_ylabel('Original', fontsize=10)
    axes[1, 0].set_ylabel('Reconstructed', fontsize=10)
    plt.suptitle('Original vs Reconstructed CIFAR10 images', y=1.02)
    plt.tight_layout()
    plt.show()


@torch.no_grad()
def show_generated_samples(model, latent_dim=20, n_images=64):
    model.eval()
    z = torch.randn(n_images, latent_dim, device=device)
    probs = model.decoder(z).view(-1, 1, 28, 28)
    samples = torch.bernoulli(probs)

    grid = make_grid(samples.cpu(), nrow=8, pad_value=1.0)
    plt.figure(figsize=(8, 8))
    plt.imshow(grid.permute(1, 2, 0).squeeze(), cmap='gray')
    plt.axis('off')
    plt.title('Generated CIFAR10 samples from latent prior')
    plt.show()


show_reconstructions(model, test_loader, n_images=10)
show_generated_samples(model, latent_dim=latent_dim, n_images=64)

In [ ]:
@torch.no_grad()
def preprocess_for_fid(images):
    # FID expects 3-channel images, typically uint8 in [0, 255]
    images = images.repeat(1, 3, 1, 1)
    images = F.interpolate(images, size=(299, 299), mode='bilinear', align_corners=False)
    images = (images.clamp(0, 1) * 255).to(torch.uint8)
    return images
    

@torch.no_grad()
def evaluate_ssim_and_fid(model, test_loader, latent_dim=20, num_fid_samples=None):
    model.eval()

    ssim_metric = StructuralSimilarityIndexMeasure(data_range=1.0).to(device)
    fid_metric = FrechetInceptionDistance(feature=2048).to(device)

    total_test = len(test_loader.dataset)
    if num_fid_samples is None:
        num_fid_samples = total_test

    # Real and reconstructed images for SSIM
    for x, _ in test_loader:
        x = x.to(device)
        x_flat = x.view(x.size(0), -1)
        x_prob, _, _ = model(x_flat)
        x_recon = x_prob.view(-1, 1, 28, 28)

        ssim_metric.update(x_recon, x)

    ssim_value = float(ssim_metric.compute().item())

    # FID: real test images vs generated images
    real_seen = 0
    for x, _ in test_loader:
        if real_seen >= num_fid_samples:
            break

        x = x.to(device)
        needed = min(x.size(0), num_fid_samples - real_seen)
        real_batch = x[:needed]
        fid_metric.update(preprocess_for_fid(real_batch), real=True)
        real_seen += needed

    fake_seen = 0
    while fake_seen < num_fid_samples:
        current_bs = min(BATCH_SIZE, num_fid_samples - fake_seen)
        z = torch.randn(current_bs, latent_dim, device=device)
        x_gen_prob = model.decoder(z).view(-1, 1, 28, 28)
        x_gen = torch.bernoulli(x_gen_prob)
        fid_metric.update(preprocess_for_fid(x_gen), real=False)
        fake_seen += current_bs

    fid_value = float(fid_metric.compute().item())
    return ssim_value, fid_value


# Use all 10,000 test samples by default; lower this for faster prototyping if needed.
ssim_value, fid_value = evaluate_ssim_and_fid(model, test_loader, latent_dim=latent_dim, num_fid_samples=10000)
print(f'SSIM: {ssim_value:.4f}')
print(f'FID:  {fid_value:.4f}')

In [ ]:
# Table 1: summary statistics
min_training_loss = float(np.min(history['BCE + KLD']))

table_1 = pd.DataFrame([
    {
        'Model': 'VAE (CIFAR10)',
        'Min Training Loss': min_training_loss,
        'Training Time (s)': float(training_time_sec),
        'SSIM': ssim_value,
        'FID': fid_value
    }
])

table_1 = table_1.round({
    'Min Training Loss': 4,
    'Training Time (s)': 2,
    'SSIM': 4,
    'FID': 4
})

print('Table 1. Training and Evaluation Summary')
display(table_1)